In [6]:
import os

# Set working directory to project root regardless of where notebook is opened from.
# This ensures all relative paths (data/, results/, etc.) work consistently.
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("03_preprocessing.ipynb"))))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\CyrilCorp\PyCharmMiscProject\ids-iiot-deeplearning


In [8]:
import pandas as pd
import numpy as np
import os

# Load the cleaned EDA output from Day 2.
# This file has 45,055 samples, 50 numeric features, and 1 binary label.
# All identifiers, string columns, and redundant correlated features
# have already been removed in the EDA stage.
df = pd.read_csv("data/processed/cleaned_eda.csv")

print(f"Loaded shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nAny nulls: {df.isnull().sum().sum()}")
print(f"\nDtypes:\n{df.dtypes.value_counts()}")

Loaded shape: (45055, 51)

Label distribution:
label
0    27360
1    17695
Name: count, dtype: int64

Any nulls: 0

Dtypes:
float64    34
int64      17
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features from the target label.
# X contains the 50 network traffic features.
# y contains the binary label (1=attack, 0=benign).
X = df.drop(columns=['label'])
y = df['label']

print(f"Features shape: {X.shape}")
print(f"Label shape: {y.shape}")
print(f"\nFeature value ranges before scaling:")
print(X.describe().loc[['min', 'max', 'mean', 'std']].T.head(10))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Apply StandardScaler to normalize all features to mean=0, std=1.
# WHY WE SCALE:
# Neural networks use gradient descent to learn — the step size (gradient) is influenced by the magnitude of the input values. When features have  vastly different scales, the optimizer spends most of its effort navigating the large-scale features and barely adjusts for small-scale ones.
# Example from our data:
#   network_fragmented-packets: 0 to 34,676 (huge range)
#   network_ip-flags_avg:       0 to 2      (tiny range)
# Without scaling, the model effectively ignores ip-flags_avg because its gradient contribution is dwarfed by fragmented-packets.
#
# WHY STANDARDSCALER SPECIFICALLY:
# StandardScaler transforms each feature to mean=0 and std=1.
# This works well for normally distributed or skewed data (which our network traffic features are). Every feature ends up on equal footing.
#
# IMPORTANT — DATA LEAKAGE NOTE:
# Here we are fitting the scaler on ALL data just to verify the transformation looks correct. In the actual preprocessing pipeline below, we will:
#   1. Split the data FIRST into train/val/test
#   2. Fit the scaler ONLY on training data
#   3. Transform val and test using the training scaler
# This prevents data leakage — the model should never have any information
# about the val/test sets during training, including their scaling statistics.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"After scaling — first 10 features:")
print(f"Mean (should be ~0): {X_scaled[:, :10].mean(axis=0).round(4)}")
print(f"Std  (should be ~1): {X_scaled[:, :10].std(axis=0).round(4)}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train (70%), validation (15%), and test (15%) sets.
#
# WHY THREE SPLITS:
# - Training set (70%): what the model learns from
# - Validation set (15%): used during training to tune hyperparameters
#   and monitor for overfitting — model never trains on this data
# - Test set (15%): held out entirely until final evaluation.
#   Gives an honest measure of real-world performance on unseen data.
#
# WHY STRATIFIED:
# stratify=y ensures each split maintains the same 39/61 attack/benign
# ratio as the full dataset. Without this, one split could end up with
# a very different class balance by random chance, making results inconsistent.
#
# WHY RANDOM_STATE=42:
# Fixes the random seed so splits are identical every time the notebook runs.
# Essential for reproducibility — anyone rerunning this gets the same results.

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Second split: split the 30% temp into 15% val and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train size:      {X_train.shape} | Attack ratio: {y_train.mean():.2%}")
print(f"Validation size: {X_val.shape}   | Attack ratio: {y_val.mean():.2%}")
print(f"Test size:       {X_test.shape}   | Attack ratio: {y_test.mean():.2%}")

In [ ]:
# Fit StandardScaler on TRAINING data only, then transform all three splits.
#
# WHY FIT ON TRAINING ONLY:
# If we fit the scaler on the full dataset (as we did above for verification),the val and test sets influence the scaling statistics (mean and std).
# This is data leakage — the model indirectly "sees" val/test information before evaluation, giving artificially optimistic results.
#
# CORRECT APPROACH:
# 1. Fit scaler on X_train only — learns mean/std from training data alone
# 2. Transform X_train using those statistics
# 3. Transform X_val and X_test using the SAME training statistics
# This simulates real deployment where you scale new incoming data using
# statistics learned from your training set, not the new data itself.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"Train scaled — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Val scaled   — mean: {X_val_scaled.mean():.4f},   std: {X_val_scaled.std():.4f}")
print(f"Test scaled  — mean: {X_test_scaled.mean():.4f},  std: {X_test_scaled.std():.4f}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle the 39/61 class imbalance.
#
# WHY CLASS WEIGHTS:
# During training, the model sees more benign samples (61%) than attack (39%). Without correction it learns to favor predicting benign because that's the safer bet statistically — leading to high accuracy but poor attack detection.
#
# Class weighting penalizes the model more heavily for misclassifying the minority class (attacks). Concretely: if attack weight = 1.55, every attack misclassification counts 1.55x more in the loss function than a benign misclassification.
#
# This is passed directly to sklearn models as class_weight='balanced' and to Keras models as the class_weight parameter in model.fit().
classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced',
                               classes=classes,
                               y=y_train)
class_weights = dict(zip(classes, weights))

print(f"Class weights: {class_weights}")
print(f"\nInterpretation:")
print(f"  Benign (0) weight:  {class_weights[0]:.4f}")
print(f"  Attack (1) weight:  {class_weights[1]:.4f}")

In [ ]:
import joblib

# WHY SAVE THE SCALER:
# The scaler must be saved alongside the model. When the model is deployed and receives new network traffic, that data must be scaled using the exact same statistics (mean/std) learned from the training set. Saving the scaler ensures this is always consistent — never refit on new data.
#
# WHY SAVE CLASS WEIGHTS:
# Saved so every model notebook can load them without recomputing, ensuring all models use identical weighting during training.

os.makedirs("data/processed", exist_ok=True)

# Save feature arrays
np.save("data/processed/X_train.npy", X_train_scaled)
np.save("data/processed/X_val.npy",   X_val_scaled)
np.save("data/processed/X_test.npy",  X_test_scaled)

# Save labels
np.save("data/processed/y_train.npy", y_train.values)
np.save("data/processed/y_val.npy",   y_val.values)
np.save("data/processed/y_test.npy",  y_test.values)

# Save scaler for deployment/inference use
joblib.dump(scaler, "data/processed/scaler.pkl")

# Save class weights
np.save("data/processed/class_weights.npy",
        np.array([class_weights[0], class_weights[1]]))

print("Saved:")
print(f"  X_train: {X_train_scaled.shape}")
print(f"  X_val:   {X_val_scaled.shape}")
print(f"  X_test:  {X_test_scaled.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_val:   {y_val.shape}")
print(f"  y_test:  {y_test.shape}")
print(f"  scaler.pkl saved")
print(f"  class_weights.npy saved")

In [14]:
#Summary:
#31,538 training samples — what the models learn from
#6,759 test samples — locked away until final evaluation
#6,758 validation samples — used during training to monitor performance
#scaler.pkl — for consistent scaling at inference time
#class_weights.npy — loaded by every model notebook

Saved:
  X_train: (31538, 50)
  X_val:   (6758, 50)
  X_test:  (6759, 50)
  y_train: (31538,)
  y_val:   (6758,)
  y_test:  (6759,)
  scaler.pkl saved
  class_weights.npy saved
